In [ ]:
"""
dataset_generation_benchmark.ipynb
Prepares two REAL/externally-sourced benchmark causal-discovery datasets
with exact ground truth, saved into this notebook's own directory
(cg_evaluations/datasets/benchmark/).

  1. Sachs                  -- real protein-signaling flow cytometry data
                                (Sachs et al. 2005), loaded via the `cdt`
                                (Causal Discovery Toolbox) package's
                                bundled dataset loader.
  2. Nonlinear + confounder -- synthetic-but-published time-series
                                benchmark, ported from TimeGraph (Ferdous,
                                Hossain, Gani -- KDD 2025), Codes/c1c.py.

Unlike dataset_generation_synthetic.ipynb (self-authored generators), both
datasets here originate from external sources -- each generation function
below documents exactly where its data/ground-truth came from, with
citations. See dataset_generation.txt (one level up) for the full
citation/resource writeup.
"""
import os
import json
import numpy as np
import pandas as pd

# datasets/benchmark/ -- this notebook's own directory
DATASETS_DIR = "."
os.makedirs(DATASETS_DIR, exist_ok=True)
print(f"Saving into: {os.path.abspath(DATASETS_DIR)}")


In [ ]:
def save_dataset(name, df, true_edges, meta):
    """Saves one dataset's three files into DATASETS_DIR, in the format
    the evaluation notebooks (evaluate_cg.ipynb / evaluate_cg_bench.ipynb)
    expect via their own load_dataset_and_truth() helper:
       {name}.csv            -- the data
       {name}_truth.json     -- ground-truth edges, used for scoring
       {name}_meta.json      -- parameters, citations, verification report
    """
    df.to_csv(f"{DATASETS_DIR}/{name}.csv", index=False)
    with open(f"{DATASETS_DIR}/{name}_truth.json", "w") as f:
        json.dump(true_edges, f, indent=2)
    meta_serializable = {k: v for k, v in meta.items()}  # meta has no numpy here
    with open(f"{DATASETS_DIR}/{name}_meta.json", "w") as f:
        json.dump(meta_serializable, f, indent=2)
    status = "PASS" if meta["verification"]["pass"] else "FAIL"
    print(f"[{status}] {name}: {df.shape[0]}x{df.shape[1]}, "
          f"{len(true_edges)} true edges  -> saved 3 files")


In [ ]:
def generate_sachs():
    """
    Real protein-signaling flow cytometry dataset: 11 phosphoproteins/
    phospholipids, 7466 single-cell measurements. A canonical benchmark
    in the causal-discovery literature -- unlike every other dataset in
    this suite, it is REAL biological data, not synthetic.

    Reference: Sachs, K., Perez, O., Pe'er, D., Lauffenburger, D.A.,
    Nolan, G.P. (2005). "Causal Protein-Signaling Networks Derived from
    Multiparameter Single-Cell Data." Science, 308(5721), 523-529.
    https://doi.org/10.1126/science.1105809

    Data + consensus ground-truth graph loaded directly via the `cdt`
    (Causal Discovery Toolbox) package's bundled dataset loader --
    cdt.data.load_dataset('sachs') -- which ships this exact data and
    graph as package resources (no network access, no local file
    dependency on any other folder in this repo). Confirmed by direct
    comparison against this project's earlier prepared copy of the same
    data: calling cdt.data.load_dataset('sachs') reproduces the exact
    same 7466x11 data and the same 18-edge consensus graph.

    Returns (df, true_edges, meta). true_edges: [{'cause','effect'}]
    """
    from cdt.data import load_dataset

    df, graph = load_dataset('sachs')
    true_edges = [{"cause": u, "effect": v} for u, v in graph.edges()]

    meta = {
        "name": "sachs", "kind": "iid_real",
        "d": df.shape[1], "n_rows": df.shape[0],
        "columns": df.columns.tolist(),
        "n_edges": len(true_edges),
        "reference": "Sachs et al. (2005), 'Causal Protein-Signaling "
                      "Networks Derived from Multiparameter Single-Cell "
                      "Data', Science 308(5721):523-529, "
                      "doi:10.1126/science.1105809. Data + consensus graph "
                      "loaded via cdt.data.load_dataset('sachs') (Causal "
                      "Discovery Toolbox package).",
        "note": "A REAL (not synthetic) biological dataset -- expect "
                "meaningfully lower scores here than on the synthetic "
                "benchmarks. That is a known, published property of Sachs "
                "as a causal-discovery benchmark (real biological noise, "
                "unmodeled regulatory mechanisms), not a bug in any "
                "method tested against it.",
    }
    meta["verification"] = {
        "no_constant_column": bool((df.std().to_numpy() > 1e-8).all()),
        "no_nan": bool(not df.isna().any().any()),
        "rows_ok": len(df) >= 30,
        "n_true_edges": len(true_edges),
    }
    meta["verification"]["pass"] = all([
        meta["verification"]["no_constant_column"],
        meta["verification"]["no_nan"], meta["verification"]["rows_ok"],
    ])
    return df, true_edges, meta


# --- Sachs (real, IID) ---
sachs_df, sachs_edges, sachs_meta = generate_sachs()
save_dataset("sachs", sachs_df, sachs_edges, sachs_meta)
print("  verification:", sachs_meta["verification"])
print("  true edges:", [(e['cause'], e['effect']) for e in sachs_edges])


In [ ]:
def get_nonlinear_equations(n_vars: int, max_lag: int):
    """
    Structural equations for the nonlinear + confounder benchmark, ported
    from TimeGraph's Codes/c1c.py get_nonlinear_equations() (Ferdous,
    Hossain, Gani -- KDD 2025). Only the (n_vars=8, max_lag=3) config used
    by this project is implemented.

    Each entry: (target, [(source, lag, nonlinearity, coefficient), ...]).
    Processing order is DELIBERATELY KEPT IDENTICAL to the reference
    (X8, X7, X6, X5, X4, X3, X2, X1) -- see generate_nonlinear_confounded's
    docstring for why this specific order matters and is preserved as-is
    rather than "corrected".
    """
    if n_vars == 8 and max_lag == 3:
        return [
            ("X8", [("X7", 0, "sin", 0.4), ("U", 0, "confounder", 0.35)]),
            ("X7", [("X6", 1, "cos", 0.35)]),
            ("X6", [("X5", 0, "sin", 0.45)]),
            ("X5", [("X4", 1, "cos", 0.4)]),
            ("X4", [("X1", 2, "cos", 0.25)]),
            ("X3", [("X4", 0, "power2", 0.35), ("X2", 3, "cos", 0.2)]),
            ("X2", [("X3", 1, "sin", 0.3)]),
            ("X1", [("X2", 0, "power3", 0.4), ("U", 0, "confounder", 0.5)]),
        ]
    raise ValueError(f"No equations defined for n_vars={n_vars}, max_lag={max_lag}")


def extract_causal_links(equations):
    """True (cause, effect, lag) edges among the X-variables only -- excludes
    the confounder U, which stays hidden from the discovery methods by
    design (see generate_nonlinear_confounded's docstring). Mirrors
    TimeGraph's own extract_causal_links(), verified to produce an
    identical edge set when run against the same equations."""
    links = []
    for target, terms in equations:
        for source, lag, func, coef in terms:
            if source == "U":
                continue
            links.append({"cause": source, "effect": target, "lag": lag,
                          "nonlinearity": func})
    return links


def generate_nonlinear_confounded(
    n_vars: int = 8,
    max_lag: int = 3,
    n_rows: int = 3000,
    noise_scale: float = 0.1,
    trend_strength: float = 0.01,
    seasonal_strength: float = 0.5,
    seasonal_period: int = 12,
    seed: int = 42,
):
    """
    Nonlinear time series with a hidden confounder U, trend, and
    seasonality. Ported from TimeGraph's NonlinearTimeSeriesGenerator
    (Codes/c1c.py, Ferdous/Hossain/Gani, KDD 2025) -- same equations, same
    coefficients, same per-timestep processing order.

    Two things verified directly against the reference source rather than
    assumed from TimeGraph's README:

    1. Irregular sampling: the README describes this category ('C1C') as
       irregular-sampling, but Codes/c1c.py's actual generator uses a
       fully regular integer time index. Reproduced faithfully here
       (regular, not irregular).

    2. Processing-order quirk: equations are evaluated in a FIXED order
       (X8, X7, X6, X5, X4, X3, X2, X1), matching the reference exactly.
       Verified directly (traced the array indexing, not inferred): two
       of the four *contemporaneous* (lag-0) edges -- X7->X8 and X5->X6
       -- have their source read as exactly 0.0 by the target's equation,
       since row t starts as zeros and the source is processed LATER in
       this fixed order, so that specific nonlinear term (e.g. X8's
       0.4*sin(X7[t]*pi/2)) contributes nothing, every timestep. The
       other two contemporaneous edges (X4->X3, X2->X1) don't have this
       problem, since X4 and X2 are processed before X3 and X1
       respectively.
       IMPORTANT: this does NOT mean X7/X8 (or X5/X6) end up uncorrelated
       -- empirically they still show real correlation (~0.58 after
       de-trending, vs ~0.12-0.14 for the unaffected pairs), because
       they're connected through other paths in this recursive network
       (X7 depends on X6[t-1]; X6 itself only gets trend+season+noise
       for the lag-0 term for the same reason, X8 gets U plus
       trend+season+noise). The quirk means the SPECIFIC direct
       nonlinear term the equation claims is dead code at runtime, not
       that the target and source are statistically independent.
       Preserved as-is (not "corrected") because the actual benchmark
       CSV already downloaded and evaluated in this project's notebooks
       was generated with this exact order -- silently fixing it here
       would make this generator produce structurally different data than
       what was already tested and reported on.

    U affects X1 (coefficient 0.5) and X8 (coefficient 0.35), both lag 0.
    U IS included as a column in the returned df for diagnostics, but
    true_edges deliberately excludes any U-involving edge -- callers
    should pass only the X* columns to run_*() so the confounding stays
    genuinely hidden from the discovery methods (matching this project's
    LPCMCI hidden-confounder benchmark convention). A fabricated direct
    X1<->X8 edge from a non-PAG method is an EXPECTED artifact of this
    design, not a bug.

    Returns (df, true_edges, meta).
    true_edges: [{'cause','effect','lag','nonlinearity'}]
    """
    rng = np.random.default_rng(seed)
    equations = get_nonlinear_equations(n_vars, max_lag)

    def noise(size=1):
        return rng.normal(0, noise_scale, size=size)

    def trend(var_idx, t):
        return trend_strength * (var_idx + 1) * 0.5 * t

    def seasonality(var_idx, t):
        phase = 2 * np.pi * var_idx / 8
        s1 = np.sin(2 * np.pi * t / seasonal_period + phase)
        s2 = 0.5 * np.cos(4 * np.pi * t / seasonal_period + phase)
        return seasonal_strength * (s1 + s2)

    def apply_nonlinearity(func, val):
        if func == "sin":
            return np.sin(val * np.pi / 2)
        if func == "cos":
            return np.cos(val * np.pi / 2)
        if func == "power2":
            return val ** 2
        if func == "power3":
            return val ** 3
        raise ValueError(f"unknown nonlinearity {func!r}")

    X = np.zeros((n_rows, n_vars))
    U = np.zeros(n_rows)

    # Initialize the first max_lag steps with noise only (matches reference)
    for i in range(max_lag):
        X[i] = noise(n_vars)
        U[i] = noise(1)[0]

    for t in range(max_lag, n_rows):
        U[t] = noise(1)[0]
        # Fixed processing order, matching the reference exactly -- see
        # docstring above for why this specific order is preserved as-is.
        for target, terms in equations:
            var_idx = int(target[1:]) - 1
            val = 0.0
            for source, lag, func, coef in terms:
                if source == "U":
                    val += coef * U[t]
                else:
                    source_idx = int(source[1:]) - 1
                    src_val = X[t, source_idx] if lag == 0 else X[t - lag, source_idx]
                    val += coef * apply_nonlinearity(func, src_val)
            val += trend(var_idx, t) + seasonality(var_idx, t) + noise(1)[0]
            X[t, var_idx] = val

    cols = [f"X{i+1}" for i in range(n_vars)]
    df = pd.DataFrame(X, columns=cols)
    df["U"] = U
    df["time"] = np.arange(n_rows)

    true_edges = extract_causal_links(equations)

    meta = {
        "name": "nonlinear_confounded",
        "kind": "timeseries_nonlinear_confounded",
        "source": "Ported from TimeGraph (Ferdous, Hossain, Gani -- KDD 2025), "
                   "Codes/c1c.py",
        "d": n_vars, "n_rows": n_rows, "max_lag": max_lag, "seed": seed,
        "columns_observed_by_methods": cols,
        "confounder_column": "U",
        "note_irregular_sampling": (
            "TimeGraph's README calls this category ('C1C') irregular-sampling, "
            "but Codes/c1c.py's actual generator uses a fully regular time "
            "index -- reproduced faithfully here (regular), verified against "
            "the source rather than assumed from the README."
        ),
        "note_processing_order": (
            "Equations evaluated in a fixed order (X8,X7,X6,X5,X4,X3,X2,X1), "
            "matching the reference exactly. X7->X8 and X5->X6 are "
            "contemporaneous edges whose source is processed AFTER the "
            "target in this order, so the target's equation reads exactly "
            "0.0 for that source that timestep -- verified directly by "
            "tracing the array indexing, not inferred. This makes that one "
            "direct nonlinear term dead code every timestep, but does NOT "
            "make the pair uncorrelated: X7/X8 and X5/X6 still show real "
            "correlation (~0.58 de-trended) via other paths in the network "
            "(e.g. X7's own real dependency on X6[t-1]). Reference-generator "
            "quirk, preserved intentionally rather than corrected (see "
            "function docstring)."
        ),
        "confounder_design": (
            "U affects X1 (coef 0.5) and X8 (coef 0.35), both lag 0. U is "
            "present as a column for diagnostics but true_edges excludes any "
            "U-involving edge; pass only the X* columns to run_*() so the "
            "confounding stays hidden. A fabricated direct X1<->X8 edge from "
            "a non-PAG method is expected, not a bug."
        ),
        "nonlinearity": "sin/cos (bounded periodic) + power2/power3 terms, "
                        "plus additive per-variable linear trend and seasonality.",
    }

    x_cols_std_ok = bool((df[cols].std().to_numpy() > 1e-8).all())
    no_nan = bool(not df.isna().any().any())
    rows_ok = len(df) >= 30
    meta["verification"] = {
        "no_constant_column": x_cols_std_ok,
        "no_nan": no_nan,
        "rows_ok": rows_ok,
        "n_true_edges": len(true_edges),
        "confounded_pair": ["X1", "X8"],
    }
    meta["verification"]["pass"] = all([x_cols_std_ok, no_nan, rows_ok])

    return df, true_edges, meta


# --- Nonlinear + confounder (time series) ---
nc_df, nc_edges, nc_meta = generate_nonlinear_confounded()
save_dataset("nonlinear_confounded", nc_df, nc_edges, nc_meta)
print("  verification:", nc_meta["verification"])
print("  true edges:", [(e['cause'], e['effect'], e['lag']) for e in nc_edges])
